In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

In [ ]:
import exciting_environments as excenvs

import dmpe
from dmpe.models.models import NeuralEulerODEPendulum, NeuralODEPendulum, NeuralEulerODE, NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env
from dmpe.models.model_training import ModelTrainer
from dmpe.excitation.excitation_utils import loss_function, Exciter

from dmpe.utils.density_estimation import (
    update_density_estimate_single_observation, update_density_estimate_multiple_observations, DensityEstimate, get_uniform_target_distribution
)
from dmpe.utils.signals import aprbs
from dmpe.evaluation.plotting_utils import (
    plot_sequence, append_predictions_to_sequence_plot, plot_sequence_and_prediction, plot_model_performance
)
from dmpe.evaluation.experiment_utils import (
    get_experiment_ids, load_experiment_results, quick_eval, evaluate_experiment_metrics, evaluate_algorithm_metrics, load_all_experiment_results
)
from dmpe.utils.density_estimation import select_bandwidth
from dmpe.evaluation.experiment_utils import default_jsd, default_ae, default_mcudsa, default_ksfc, default_df

In [ ]:
from dmpe.data_management import DataPaths
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence, valid_space_grid
from dmpe.utils.sets.shared import check_in_set, load_results, DiscretizedSet, SlicedSet, load_discretized_set, save_discretized_set

In [ ]:
algo = "dmpe"
system = "cart_pole"

results = load_all_experiment_results(DataPaths().se_cs_experiments / algo / system, None)
print(len(results))
for result in results:
    print(result["exp_id"], result["params"]["seed"])

In [ ]:
result = results[0]
result.keys()

In [ ]:
default_jsd(result["observations"], result["actions"], points_per_dim=15, bandwidth=select_bandwidth(2, 5, 15, 0.1).item())

In [ ]:
S_xu = load_discretized_set(DataPaths().reach_ci_experiments / "cart_pole_S_xu_8744b5d5-30e6-4b.json")

In [ ]:
fig, axs = S_xu.visualize(use_contourf=True)
data_points = jnp.concatenate([result["observations"][:-1], result["actions"]], axis=-1)
n_features = data_points.shape[-1]
for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.1, c="r")

In [ ]:
points_per_dim = 11

check_in_S_xu = partial(check_in_set, grid=S_xu.grid, mask=S_xu.mask)

target_distribution_p = get_uniform_target_distribution(
    dim=data_points.shape[-1],
    points_per_dim=points_per_dim,
    bandwidth=0.08,
    grid_extend=1.0,
    consider_action_distribution=True,
    penalty_function=lambda x, u: jnp.logical_not(check_in_S_xu(jnp.concatenate([x, u], axis=-1))),
    obs_dim=4,
    act_dim=1,
)

In [ ]:
default_jsd(data_points[:, :-1], data_points[:, -1:], points_per_dim=11, bandwidth=0.08, target_distribution=target_distribution_p, ca=True)

- build versions of the other metrics that respect the feasible set

In [ ]:
points_per_dim = 20

In [ ]:
from dmpe.evaluation.utils import valid_space_grid

In [ ]:
support_points_ = valid_space_grid(
    constraint_function=lambda z: jnp.logical_not(check_in_S_xu(z)),
    data_dim=5,
    points_per_dim=points_per_dim,
    min_value=-1,
    max_value=1,
)

In [ ]:
print(partial(default_mcudsa, points_per_dim=None, support_points=support_points)(data_points[:, :-1], data_points[:, -1:]))
print(partial(default_ksfc, points_per_dim=None, variance=0.1, eps=1e-6, support_points=support_points)(data_points[:, :-1], data_points[:, -1:]))

In [ ]:
print(partial(default_mcudsa, points_per_dim=None, support_points=support_points_)(data_points[:, :-1], data_points[:, -1:]))
print(partial(default_ksfc, points_per_dim=None, variance=0.1, eps=1e-6, support_points=support_points_)(data_points[:, :-1], data_points[:, -1:]))

In [ ]:
support_points = support_points_

In [ ]:
from dmpe.evaluation.plotting_utils import plot_feature_combinations
plot_feature_combinations(support_points, labels=["d", "v", "theta", "omega", "F"]);
plt.show()
plot_feature_combinations(support_points, labels=["d", "v", "theta", "omega", "F"], mode="contourf", points_per_dim=10, bandwidth=0.08);
plt.show()

In [ ]:
# dim = 5
# bounds = (-1, 1)
# support_points = dmpe.utils.density_estimation.build_grid(dim, low=bounds[0], high=bounds[1], points_per_dim=points_per_dim)

# constr_function = check_in_S_xu
# valid_grid_point = jax.vmap(constr_function, in_axes=0)(support_points) == 0
# support_points = support_points[jnp.where(valid_grid_point == True)]

In [ ]:
from dmpe.utils.metrics import blockwise_mcudsa, blockwise_ksfc

In [ ]:
default_ksfc

In [ ]:
print(partial(default_mcudsa, points_per_dim=11)(data_points[:, :-1], data_points[:, -1:]))
print(partial(default_mcudsa, points_per_dim=None, support_points=support_points)(data_points[:, :-1], data_points[:, -1:]))

blockwise_mcudsa(
    data_points,
    support_points,
)

In [ ]:
print(partial(default_ksfc, points_per_dim=11, variance=0.1, eps=1e-6)(data_points[:, :-1], data_points[:, -1:]))
print(partial(default_ksfc, points_per_dim=None, variance=0.1, eps=1e-6, support_points=support_points)(data_points[:, :-1], data_points[:, -1:]))
blockwise_ksfc(
    data_points,
    support_points,
    variances=jnp.ones([5]) * 0.1,
    eps=1e-6,
)

In [ ]:
test_ksfc = partial(blockwise_ksfc, support_points=support_points,variances=jnp.ones([5]) * 0.1, eps=1e-6)
test_mcudsa = partial(blockwise_mcudsa, support_points=support_points)

- what about AE and DF?
- DF should def be possible?
- I do not think AE is possible?

In [ ]:
default_df

In [ ]:
data_points.shape

In [ ]:
from dmpe.utils.metrics import diced_fill, check_points_to_grid
from dmpe.utils.density_estimation import build_grid

In [ ]:
points_per_dim = 7
dim = data_points.shape[-1]

support_points = build_grid(dim, low=-1, high=1, points_per_dim=points_per_dim)
support_spacing = jnp.abs(support_points[0] - support_points[1])[-1] / 2

diced_fill(data_points, support_points, support_spacing)

In [ ]:
partial(default_df, points_per_dim=None, support_points=support_points, support_spacing=support_spacing)(data_points[:, :-1], data_points[:, -1:])
partial(default_df, points_per_dim=None, support_points=support_points_, support_spacing=support_spacing)(data_points[:, :-1], data_points[:, -1:])

In [ ]:
support_points_ = valid_space_grid(
    constraint_function=lambda z: jnp.logical_not(check_in_S_xu(z)),
    data_dim=5,
    points_per_dim=points_per_dim,
    min_value=-1,
    max_value=1,
)

In [ ]:
diced_fill(data_points, support_points_, support_spacing)

In [ ]:
support_points_.shape

In [ ]:
support_points.shape

In [ ]:
fig, axs = S_xu.visualize(use_contourf=True)
n_features = data_points.shape[-1]
for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(support_points[..., i], support_points[..., j], s=0.1, c="r")
        axs[j, i].scatter(support_points_[..., i], support_points_[..., j], s=0.1, c="b")

plot diced fill: Where did you do this already?

In [ ]:
grid = support_points
out = jnp.any(check_points_to_grid(support_points, data_points, support_spacing), axis=0)

filled_set = DiscretizedSet(grid=grid, mask=out, unflattened_shape=tuple([points_per_dim] * dim))
fig, axs = filled_set.visualize()
plt.show()

- that seems pretty bad: Is this really realistic? 1 is 100% filled 0 is 0% filled

## Eval plot reproduction:

In [ ]:
from dmpe.evaluation.experiment_utils import extract_metrics_over_timesteps, extract_metrics_over_timesteps_via_interpolation

In [ ]:
full_column_width = 18.2
half_column_width = 8.89

import matplotlib.ticker as ticker

def custom_formatter(val, pos):
    if val < 1.0:
        return rf"${val:.2f}$"  
    else:
        return rf"${val:.1f}$"  

def plot_metrics_by_sequence_length_for_all_algos(data_per_algo, lengths, algo_names, use_log=False, plot_log=False):
    assert len(data_per_algo) == len(algo_names), "Mismatch in number of algo results and number of algo names"

    metric_keys = data_per_algo[0].keys()

    fig, axs = plt.subplots(max(2, len(metric_keys)), figsize=(half_column_width, 0.7 * 11 / 4 * max(2, len(metric_keys))), sharex=True) # figsize=(19, 18)
    colors = plt.rcParams["axes.prop_cycle"]()

    for algo_name, data in zip(algo_names, data_per_algo):
        c = next(colors)["color"]
        if c == '#d62728':
            c = next(colors)["color"]

        for metric_idx, metric_key in enumerate(metric_keys):

            mean = jnp.nanmean(jnp.log(data[metric_key]), axis=0) if use_log else jnp.nanmean(data[metric_key], axis=0)
            std = jnp.nanstd(jnp.log(data[metric_key]), axis=0) if use_log else jnp.nanstd(data[metric_key], axis=0)

            if algo_name=="$\mathrm{DMPE}$":
                style = "dashed"
            elif algo_name=="$\mathrm{iGOATS}$":
                style = "dashdot"
            else:
                style=None
            
            axs[metric_idx].plot(
                lengths,
                mean, 
                label=algo_name if metric_idx == 0 else None,
                color=c,
                #linestyle='dashed' if algo_name=="$\mathrm{DMPE}$" else None,
                linewidth=2.5,
                linestyle=style,
            )
            axs[metric_idx].fill_between(
                lengths,
                mean - std,
                mean + std,
                color=c,
                alpha=0.1,
            )

    if plot_log:
        for ax in axs:
            ax.set_yscale('log', base=10)

        for idx_y, ax in enumerate(axs[:-1]):
            ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
            ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
            ax.yaxis.set_major_locator(ticker.LogLocator(numticks=2))

            if idx_y > 0:
                ax.yaxis.set_minor_locator(ticker.LogLocator(subs=(0.5,), numticks=4))
            else:
                ax.yaxis.set_minor_locator(ticker.LogLocator(subs=(0.3, 0.6), numticks=4))
            #ax.yaxis.set_minor_locator(ticker.LogLocator(subs="auto"))
    
    for idx, metric_key in enumerate(metric_keys):
        axs[idx].set_ylabel(f"$\mathcal{{L}}_\mathrm{{{metric_key.upper()}}}$")

    axs[-1].set_xlabel("$k$")
    axs[-1].set_xlim(lengths[0], lengths[-1])

    [ax.grid(True, which="both", alpha=0.3) for ax in axs]
    
    legend = fig.legend(
        prop={'size': 5 * 2.54},
        framealpha=0.5,
        loc="center",
        bbox_to_anchor=(0.525, -0.02),
        fancybox=True,
        shadow=False,
        ncol=len(algo_names)
    )

    plt.subplots_adjust(hspace=0.02)
    
    plt.tight_layout(pad=0.05)

    [ax.tick_params(axis="y", direction='in') for ax in axs]
    [ax.tick_params(axis="x", direction='in') for ax in axs]
    # [ax.yaxis.set_major_locator(plt.MaxNLocator(3)) for ax in axs]

    fig.align_ylabels(axs)

    return fig

In [ ]:
def extract_results(lengths, raw_results_path, algo_names, interpolate_to_lengths, system_name, metrics=None, extra_folders=None):

    all_results_by_metric = {}
    
    for (algo_name, use_interpolation) in zip(algo_names, interpolate_to_lengths):
        full_results_path = raw_results_path / pathlib.Path(algo_name) / pathlib.Path(system_name)
        full_results_path = full_results_path / pathlib.Path(extra_folders) if extra_folders is not None else full_results_path

        print("Extract results for", algo_name, "\n at", full_results_path)

        if not use_interpolation:
            all_results_by_metric[algo_name] = extract_metrics_over_timesteps(
                experiment_ids=get_experiment_ids(full_results_path),
                results_path=full_results_path,
                lengths=lengths,
                metrics=metrics,
            )
        else:
            all_results_by_metric[algo_name] = extract_metrics_over_timesteps_via_interpolation(
                experiment_ids=get_experiment_ids(full_results_path),
                results_path=full_results_path,
                target_lengths=lengths,
                metrics=metrics,
            )
        print("\n")
    return all_results_by_metric


def test_jsd(observations, actions):
    if observations.shape[0] == actions.shape[0] + 1:
        observations = observations[0:-1, :]

    data_points = jnp.concatenate([observations, actions], axis=-1)
    return jsd_(data_points)

In [ ]:
#lengths = jnp.linspace(1000, 15000, 3, dtype=jnp.int32)
lengths = jnp.linspace(1000, 15000, 15, dtype=jnp.int32)
lengths

In [ ]:
system_name = "cart_pole"

all_cart_pole_results_by_metric = extract_results(
    lengths=lengths,
    raw_results_path=DataPaths().se_cs_experiments,
    algo_names=["dmpe", "sgoats", "perfect_model_dmpe", "igoats", "random_walk"],
    interpolate_to_lengths=[False, False, False, False, False], #False],
    system_name=system_name,
    extra_folders=None,
    metrics={
        "jsd": partial(default_jsd, points_per_dim=11, bandwidth=0.08, target_distribution=target_distribution_p, ca=True),
        "mcudsa": partial(default_mcudsa, points_per_dim=None, support_points=support_points),
        "ksfc": partial(default_ksfc, points_per_dim=None, variance=0.1, eps=1e-6, support_points=support_points),
    }
)

In [ ]:
print(all_cart_pole_results_by_metric.keys())

pm_dmpe_results_by_metric = all_cart_pole_results_by_metric["perfect_model_dmpe"]
dmpe_results_by_metric = all_cart_pole_results_by_metric["dmpe"]
sgoats_results_by_metric = all_cart_pole_results_by_metric["sgoats"]#["interp"]
igoats_results_by_metric = all_cart_pole_results_by_metric["igoats"]#["interp"]
random_walk_results_by_metric = all_cart_pole_results_by_metric["random_walk"]

plot_metrics_by_sequence_length_for_all_algos(
    data_per_algo=[
        pm_dmpe_results_by_metric,
        dmpe_results_by_metric,
        sgoats_results_by_metric,
        igoats_results_by_metric,
        random_walk_results_by_metric
    ],
    lengths=lengths,
    algo_names=[
        "$\mathrm{PM-DMPE}$",
        "$\mathrm{DMPE}$",
        "$\mathrm{sGOATS}$",
        "$\mathrm{iGOATS}$",
        "$\mathrm{random-walk}$",
    ],
    use_log=False,
    plot_log=True
);

In [ ]:
print(all_cart_pole_results_by_metric.keys())

pm_dmpe_results_by_metric = all_cart_pole_results_by_metric["perfect_model_dmpe"]
dmpe_results_by_metric = all_cart_pole_results_by_metric["dmpe"]
sgoats_results_by_metric = all_cart_pole_results_by_metric["sgoats"]#["interp"]
igoats_results_by_metric = all_cart_pole_results_by_metric["igoats"]#["interp"]
random_walk_results_by_metric = all_cart_pole_results_by_metric["random_walk"]

plot_metrics_by_sequence_length_for_all_algos(
    data_per_algo=[
        pm_dmpe_results_by_metric,
        dmpe_results_by_metric,
        sgoats_results_by_metric,
        igoats_results_by_metric,
        random_walk_results_by_metric
    ],
    lengths=lengths,
    algo_names=[
        "$\mathrm{PM-DMPE}$",
        "$\mathrm{DMPE}$",
        "$\mathrm{sGOATS}$",
        "$\mathrm{iGOATS}$",
        "$\mathrm{random-walk}$",
    ],
    use_log=False,
    plot_log=True
);
plt.savefig(f"testitest.pdf", bbox_inches='tight')

## Stuff:

In [ ]:
data_evaluator = DataEvaluator(
    constraint_function=lambda x: 0.0,
    data_dim=5,
    points_per_dim=points_per_dim,
)
jsd = data_evaluator.metrics["jsd"]
jsd(data_points) # <- full space

In [ ]:
jnp.mean((target_distribution.p - target_distribution_p)**2)

In [ ]:
# build the target_distribution based on the reachable set:

constraint_function = 

data_evaluator = DataEvaluator(
    constraint_function=constraint_function,
    data_dim=5,
    points_per_dim=points_per_dim,
)

jsd = data_evaluator.metrics["jsd"]
jsd(data_points) # <- only valid support

In [ ]:
constraint_data_space_grid = valid_space_grid(constraint_function, 5, points_per_dim, -1, 1)
target_distribution = DensityEstimate(
    p=jsd.generate_distribution(
        constraint_data_space_grid,
        valid_space_grid(lambda x: 0, 5, points_per_dim, -1, 1),
        bandwidth=0.08,
    ),
    z_g=valid_space_grid(lambda x: 0, 5, points_per_dim, -1, 1),
    bandwidth=0.08,
    n_observations=None
)
jsd_ = JensenShannonDivergence(
    target_distribution.z_g,
    bandwidth=target_distribution.bandwidth,
    target_distribution=target_distribution.p
)
jsd_(data_points) # <- full space but target based on valid support

In [ ]:
jsd_

In [ ]:
from dmpe.evaluation.plotting_utils import plot_feature_combinations

In [ ]:
plot_feature_combinations(data_points, labels=["d", "v", "theta", "omega", "F"]);

In [ ]:
plot_feature_combinations(constraint_data_space_grid, labels=["d", "v", "theta", "omega", "F"], mode="contourf", points_per_dim=11, bandwidth=0.08);

In [ ]:
plot_feature_combinations(
    data_points,
    labels=["d", "v", "theta", "omega", "F"],
    mode="contourf",
    points_per_dim=11,
    bandwidth=0.08
);